In [2]:
# CHARM
!mkdir CHARM
!cd CHARM
!git init
!git remote add origin https://github.com/opendatalab/CHARM.git
!git config core.sparseCheckout true
!echo "data/CHARM/reasoning/" >> .git/info/sparse-checkout
!git pull origin main

Reinitialized existing Git repository in E:/香港中文大学深圳CUHKSZ/MDS 5110 Natural Language Processing/Assignment/Ass3/.git/


error: remote origin already exists.


Already up to date.

From https://github.com/opendatalab/CHARM
 * branch            main       -> FETCH_HEAD


In [13]:
import pandas as pd
import numpy as np
import os
filenames = os.listdir("CHARM/data/CHARM/reasoning")
subject_list = [val_file.replace('.json','') for val_file in filenames if val_file.startswith('Chinese_')]

import json

def transform_data(subject):
    # 解析原始JSON数据
    with open('CHARM/data/CHARM/reasoning/'+subject+'.json', "r", encoding="utf-8") as f:
        data = json.load(f)
    
    # 初始化结果列表
    transformed_data = []
    
    # 遍历每个例子
    for example in data['examples']:
        # 提取必要信息
        subject_name = subject  # 根据需要设置主题名称
        question = example['input']
        answer = example['target'].replace('(', '').replace(')', '')
        
        # 创建新的格式
        new_format = {
            "subject": subject_name,
            "conversations": [
                {"from": "human", "value": '以下是中国常识考试的单项选择题，请选出其中的正确答案。\n' + question},
                {"from": "gpt", "value": answer}
            ],
            "ground_truth": answer,
        }
        
        # 添加到结果列表
        transformed_data.append(new_format)
    
    return transformed_data

print('preparing dataset...')
dataset = []
for index,subject in enumerate(subject_list):
    data = transform_data(subject)
    dataset.extend(data)
    print(f"Processed {index + 1}/{len(subject_list)}: {subject} - {len(data)} samples added")

with open("CHRAM.json", "w", encoding="utf-8") as f:
    json.dump(dataset, f, ensure_ascii=False, indent=4)

preparing dataset...
Processed 1/7: Chinese_Anachronisms_Judgment - 150 samples added
Processed 2/7: Chinese_Movie_and_Music_Recommendation - 50 samples added
Processed 3/7: Chinese_Natural_Language_Inference - 100 samples added
Processed 4/7: Chinese_Reading_Comprehension - 200 samples added
Processed 5/7: Chinese_Sequence_Understanding - 100 samples added
Processed 6/7: Chinese_Sport_Understanding - 200 samples added
Processed 7/7: Chinese_Time_Understanding - 100 samples added


In [14]:
len(dataset)

900

In [2]:
# 合并DS_pre的答题结果
import re
from tqdm import tqdm
import json

# read val_data.jsonl
with open('val_data.json', 'r', encoding='utf-8') as f:
  val_data = json.load(f)

# read model_answers.json
with open('ds_pre_result.json', 'r', encoding='utf-8') as f:
  model_answers = json.load(f)
def get_ans(ans):
    match = re.findall(r'.*?([A-E]+(?:[、, ]+[A-E]+)*)', str(ans))
    if match:
        last_match = match[-1]
        return ''.join(re.split(r'[、, ，]+', last_match))
    return ''

correct_num = 0
total_num = 0
for model_answer, item in tqdm(zip(model_answers, val_data)):
  ans = get_ans(model_answer)
  if ans  == item['ground_truth']:
    correct_num += 1
  total_num += 1
  item['model_answer'] = model_answer
  item['model_choice'] = ans

print(f"ACC: {correct_num/total_num:.2%}")

357it [00:00, 1680.16it/s]

ACC: 48.18%


In [3]:
with open("ds_pre_result.json", "w", encoding="utf-8") as f:
    json.dump(val_data, f, ensure_ascii=False, indent=4)
    print(f"Results are save in ds_pre_result.json")

Results are save in ds_pre_result.json


In [2]:
import json
with open('ds_pre_result.json', 'r', encoding='utf-8') as file:
        # 加载 JSON 数据
        a = json.load(file)

# 遍历列表，移除每个字典中的'model_answer'和'model_choice'键值对
for item in a:
    item.pop('model_answer', None)
    item.pop('model_choice', None)

# 将处理后的数据保存为val_data文件
with open('val_data.json', 'w', encoding='utf-8') as file:
    json.dump(a, file, ensure_ascii=False, indent=4)

In [3]:
# 打分
import json
with open('ds_pre_result.json', 'r', encoding='utf-8') as file:
    # 加载 JSON 数据
    data = json.load(file)

correct_count = 0
total_count = len(data)

for item in data:
    ground_truth = item["ground_truth"]
    model_answer = item["model_choice"]
    if ground_truth == model_answer:
        correct_count += 1

if total_count > 0:
    accuracy = correct_count / total_count
    print(f"ds_pre正确率为: {accuracy * 100:.2f}%")
else:
    print("没有数据可供统计。")

ds_pre正确率为: 48.18%


In [5]:
# 打分
import json
with open('qwen2_pre_result.json', 'r', encoding='utf-8') as file:
    # 加载 JSON 数据
    data = json.load(file)

correct_count = 0
total_count = len(data)

for item in data:
    ground_truth = item["ground_truth"]
    model_answer = item["model_answer"][0]
    if ground_truth == model_answer:
        correct_count += 1

if total_count > 0:
    accuracy = correct_count / total_count
    print(f"qwen_pre正确率为: {accuracy * 100:.2f}%")
else:
    print("没有数据可供统计。")

qwen_pre正确率为: 70.03%
